<a href="https://colab.research.google.com/github/sde2424242424-coder/2026_spring_assignments/blob/main/webtoon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Webtoon



이 프로그램은 네이버 웹툰 사이트에서 데이터를 가져옵니다. 먼저 BeautifulSoup으로 정적 크롤링을 시도하고, 이후 Selenium으로 동적 크롤링을 수행합니다. Selenium을 사용하여 첫 번째 웹툰의 제목과 설명을 추출합니다.

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By


TARGET_URL = "https://comic.naver.com/webtoon"
SAVE_DIR = "naver_webtoon_images"


def static_check(url):
    print("=== 1. Static Crawling with BeautifulSoup ===")

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    page_title = soup.title.string.strip() if soup.title else "No title"
    print("Page title:", page_title)

    static_items = soup.select("li[class*='TripleRecommendList__item']")
    print("Cards found by BeautifulSoup:", len(static_items))


def save_image_with_selenium(img_element, folder_name):
    os.makedirs(folder_name, exist_ok=True)

    save_path = os.path.join(folder_name, "first_webtoon.png")

    if os.path.exists(save_path):
        print("Image already exists:", save_path)
        return save_path

    img_element.screenshot(save_path)
    print("Image saved:", save_path)
    return save_path


def dynamic_crawl(url):
    print("\n=== 2. Dynamic Crawling with Selenium ===")

    browser = webdriver.Chrome()
    browser.implicitly_wait(5)

    try:
        browser.get(url)

        webtoon_list = browser.find_elements(
            By.CSS_SELECTOR,
            "li[class*='TripleRecommendList__item']"
        )

        print("Cards found by Selenium:", len(webtoon_list))

        if not webtoon_list:
            print("No cards were found.")
            return

        card = webtoon_list[0]

        try:
            webtoon_title = card.find_element(By.CSS_SELECTOR, "span.text").text
        except:
            webtoon_title = "No title"

        try:
            webtoon_author = card.find_element(
                By.CSS_SELECTOR,
                "a[class*='ContentAuthor__author']"
            ).text
        except:
            webtoon_author = "No author"

        try:
            webtoon_summary = card.find_element(
                By.CSS_SELECTOR,
                "p[class*='TripleRecommendList__summary']"
            ).text
        except:
            webtoon_summary = "No description"

        try:
            webtoon_link = card.find_element(
                By.CSS_SELECTOR,
                "a[class*='ContentTitle__title_area']"
            ).get_attribute("href")
        except:
            webtoon_link = "No link"

        print("\n=== 3. First Webtoon Information ===")
        print("Title:", webtoon_title)
        print("Author:", webtoon_author)
        print("Summary:", webtoon_summary)
        print("Link:", webtoon_link)

        try:
            image_tag = card.find_element(By.CSS_SELECTOR, "img")
            print("\n=== 4. Save Poster Image ===")
            save_image_with_selenium(image_tag, SAVE_DIR)
        except:
            print("Image element not found.")

    finally:
        browser.quit()


def main():
    static_check(TARGET_URL)
    dynamic_crawl(TARGET_URL)


if __name__ == "__main__":
    main()

=== 1. Static Crawling with BeautifulSoup ===
Page title: 네이버 웹툰
Cards found by BeautifulSoup: 0

=== 2. Dynamic Crawling with Selenium ===
Cards found by Selenium: 3

=== 3. First Webtoon Information ===
Title: 공작님의 아이만 필요합니다
Author: 라뇽 / Roal / 백단
Summary: ‘과거로 돌아갈 수 있다면, 그 사람을 사랑하지 않을 텐데.’ 회귀 후, 전남편과의 두 번째 결혼은 오직 아이를 위해서였다. 그렇게 기적처럼 얻은 두 번째 생은 사랑을 갈구하던 전생과 달리 오로지 아이를 다시 만나겠다는 일념하에 그와의 결혼을 선택했다. “딱 일 년만 나와의 결혼을 유지해줘요.” 그 대가는 그가 원하는 대로 10년 전 사건의 비밀을 알아내는 데 협조하겠다는 것. 그렇게 계약으로 묶인 이름뿐인 부부라고 생각했는데…. “반드시 첫날밤을 치를 필요는―” “내가 원해.” 그는 전생과 달랐다.
Link: https://comic.naver.com/webtoon/list?titleId=849451

=== 4. Save Poster Image ===
Image saved: naver_webtoon_images\first_webtoon.png
